> **Deprecated:** per-epoch histories are now logged directly by 04-00 and 04-01; this diagnostic is kept for reference only.

# 04-06 — Multi-Label Loss Curves (aggregated shared MLP)

Standalone diagnostic next to ``04-00-MultiLabel-CLassifier``. The main
notebook trains the aggregated shared MLP (all 4 sites, 26,642 spectra,
10 drug heads, masked BCE) but discards the per-epoch history.

This notebook retrains the **aggregated** model once at a good configuration
and records, per epoch:

- train / val masked BCE loss
- val macro Balanced Accuracy @0.5 over known labels (ranking vs calibration)

Configuration provenance: 04's aggregated grid best was not persisted; the
default below is 07's best MLP config for Ciprofloxacin. Override ``BEST`` if
you want a different point (e.g. re-run 04-00's grid).

Outputs: ``results_loss_curves_04/04_loss_curves.pdf`` + ``history_04.csv``.

Run on Colab (GPU recommended). Set env ``SMOKE=1`` for a dry run.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    print('Running locally')
    IN_COLAB = False

if IN_COLAB:
    !pip install maldideepkit maldiamrkit --quiet

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import balanced_accuracy_score

from maldideepkit.attention.mlp import SpectralAttentionMLP
from maldideepkit.base.data import fit_input_transform, apply_input_transform
from maldiamrkit.evaluation import stratified_species_drug_split

warnings.filterwarnings("ignore", category=FutureWarning)

SMOKE = os.environ.get("SMOKE", "0") == "1"
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if IN_COLAB:
    DATA_ROOT = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet/Processed")
    REPO_ROOT = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet/Processed/Processing/Analysis")
else:
    DATA_ROOT = Path("/media/asd/home/asd/Projects/Flower/DataSet/Dryad-DataSet/Processed")
    REPO_ROOT = Path("/media/asd/home/asd/Projects/Flower/DataSet/Dryad-DataSet/Processed/Processing/Analysis")

OUT_DIR = Path("./results_loss_curves_04")
OUT_DIR.mkdir(exist_ok=True)

BEST = {"lr": 7.8e-05, "drop": 0.8}   # <-- override here if desired
DEVICE_NAME = "cuda" if torch.cuda.is_available() else "cpu"

SITES = ["DRIAMS-A", "DRIAMS-B", "DRIAMS-C", "DRIAMS-D"]
BIN_COLS = [f"bin_{i}" for i in range(6000)]
DRUGS_10 = ["Ciprofloxacin", "Gentamicin", "Amoxicillin-Clavulanic acid",
            "Piperacillin-Tazobactam", "Cefepime", "Ceftriaxone",
            "Imipenem", "Ceftazidime", "Vancomycin", "Amikacin"]

print(f"SMOKE={SMOKE} | device={DEVICE_NAME}")
print(f"BEST={BEST}")

In [ ]:
# ── Multi-label dataset + trainer (same as 04-00, with history recording) ──

class MultiLabelDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(np.nan_to_num(Y, nan=0.0), dtype=torch.float32)
        self.mask = torch.tensor(~np.isnan(Y), dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx], self.mask[idx]

# =============================================================================
# 2. MULTI-LABEL MLP CLASSIFIER  (sklearn-compatible, PyTorch backend)
# =============================================================================

class MultiLabelMaldiMLP(BaseEstimator, ClassifierMixin):
    def __init__(self, hidden_dim=512, head_dims=(256, 128), use_attention=False,
                 dropout_high=0.3, dropout_low=0.2,
                 learning_rate=1e-3, weight_decay=0.0,
                 batch_size=64, epochs=50, early_stopping_patience=10,
                 warmup_epochs=0, val_fraction=0.1,
                 random_state=42, verbose=False, device="cpu"):
        self.hidden_dim = hidden_dim
        self.head_dims = head_dims
        self.use_attention = use_attention
        self.dropout_high = dropout_high
        self.dropout_low = dropout_low
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.epochs = epochs
        self.early_stopping_patience = early_stopping_patience
        self.warmup_epochs = warmup_epochs
        self.val_fraction = val_fraction
        self.random_state = random_state
        self.verbose = verbose
        self.device = device

    def _build_model(self, input_dim, n_classes):
        return SpectralAttentionMLP(
            input_dim=input_dim, n_classes=n_classes,
            hidden_dim=self.hidden_dim, head_dims=self.head_dims,
            use_attention=self.use_attention,
            dropout_high=self.dropout_high, dropout_low=self.dropout_low)

    def fit(self, X, Y):
        np.random.seed(self.random_state)
        torch.manual_seed(self.random_state)

        self.n_labels_ = Y.shape[1]
        self.input_dim_ = X.shape[1]
        self.model_ = self._build_model(self.input_dim_, self.n_labels_).to(self.device)

        ds = MultiLabelDataset(X, Y)
        n_val = max(1, int(len(ds) * self.val_fraction))
        n_tr = len(ds) - n_val
        train_ds, val_ds = random_split(ds, [n_tr, n_val],
            generator=torch.Generator().manual_seed(self.random_state))

        train_loader = DataLoader(train_ds, batch_size=self.batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=self.batch_size * 2, shuffle=False)

        opt_cls = torch.optim.AdamW if self.weight_decay > 0 else torch.optim.Adam
        optimizer = opt_cls(self.model_.parameters(), lr=self.learning_rate,
                            weight_decay=self.weight_decay)
        warmup = max(0, self.warmup_epochs)
        t_max = max(1, self.epochs - warmup)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=t_max, eta_min=1e-6)
        criterion = nn.BCEWithLogitsLoss(reduction="none")

        best_val_loss = float("inf")
        best_state = None
        patience_counter = 0

        self.history_ = {"train_loss": [], "val_loss": [], "val_macro_balacc": []}
        for epoch in range(self.epochs):
            self.model_.train()
            train_loss = 0.0
            for xb, yb, mb in train_loader:
                xb, yb, mb = xb.to(self.device), yb.to(self.device), mb.to(self.device)
                if epoch < warmup:
                    for pg in optimizer.param_groups:
                        pg["lr"] = self.learning_rate * (epoch + 1) / warmup
                optimizer.zero_grad()
                logits = self.model_(xb)
                loss = criterion(logits, yb)
                loss = (loss * mb).sum() / mb.sum()
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            if epoch >= warmup:
                scheduler.step()

            self.model_.eval()
            val_loss = 0.0
            all_logits, all_y, all_m = [], [], []
            with torch.no_grad():
                for xb, yb, mb in val_loader:
                    xb, yb, mb = xb.to(self.device), yb.to(self.device), mb.to(self.device)
                    logits = self.model_(xb)
                    loss = criterion(logits, yb)
                    loss = (loss * mb).sum() / mb.sum()
                    val_loss += loss.item()
                    all_logits.append(logits.cpu()); all_y.append(yb.cpu()); all_m.append(mb.cpu())

            val_loss /= len(val_loader)
            train_loss /= len(train_loader)

            # macro Balanced Accuracy @0.5 over known labels (ranking vs calibration check)
            logits_cat = torch.cat(all_logits); y_cat = torch.cat(all_y); m_cat = torch.cat(all_m)
            bas = []
            for di in range(logits_cat.shape[1]):
                mask = m_cat[:, di] > 0
                if mask.sum() < 2:
                    continue
                yt = y_cat[mask, di].numpy().astype(int)
                yp = (torch.sigmoid(logits_cat[mask, di]).numpy() >= 0.5).astype(int)
                if len(np.unique(yt)) < 2:
                    continue
                bas.append(balanced_accuracy_score(yt, yp))
            val_macro_ba = float(np.mean(bas)) if bas else float("nan")

            self.history_["train_loss"].append(train_loss)
            self.history_["val_loss"].append(val_loss)
            self.history_["val_macro_balacc"].append(val_macro_ba)
            if self.verbose:
                print(f"  Epoch {epoch+1:3d}/{self.epochs}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.cpu().clone() for k, v in self.model_.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= self.early_stopping_patience:
                    if self.verbose:
                        print(f"  Early stopping at epoch {epoch+1}")
                    break

        if best_state is not None:
            self.model_.load_state_dict(best_state)
        self.model_.eval()
        self.is_fitted_ = True
        return self

    def predict_proba(self, X):
        X_t = torch.tensor(X, dtype=torch.float32).to(self.device)
        with torch.no_grad():
            logits = self.model_(X_t)
            proba = torch.sigmoid(logits).cpu().numpy()
        return proba

    def predict(self, X, thresholds=None):
        proba = self.predict_proba(X)
        if thresholds is None:
            thresholds = [0.5] * self.n_labels_
        return (proba >= np.array(thresholds)).astype(int)

In [ ]:
# ── Load aggregated data: npz cache if available, else rebuild from CSVs ──
cache_candidates = [
    Path.cwd().parent / "04-00-MultiLabel-CLassifier" / "aggregated_multilabel_data.npz",
    Path.cwd() / "aggregated_multilabel_data.npz",
    REPO_ROOT / "04-Multi-Label-mlp" / "04-00-MultiLabel-CLassifier" / "aggregated_multilabel_data.npz",
]
cache_path = next((p for p in cache_candidates if p.exists()), None)

if cache_path is not None:
    print(f"Loading cache: {cache_path}")
    cached = np.load(cache_path, allow_pickle=True)
    X_all, Y_all, sp_all = cached["X_all"], cached["Y_all"], cached["sp_all"]
else:
    print("No cache found — rebuilding from Processed CSVs (slow, ~minutes)")
    Xs, Ys, sps = [], [], []
    for site_name in SITES:
        proc_site = f"Proc_{site_name}"
        site_labels, X_samples, sp_samples = {}, {}, {}
        for drug_name in DRUGS_10:
            p = DATA_ROOT / proc_site / drug_name.replace(" ", "_") / "data.csv"
            if not p.exists():
                continue
            df = pd.read_csv(p)
            for _, row in df.iterrows():
                code = row["code"]
                if code not in X_samples:
                    X_samples[code] = row[BIN_COLS].values.astype("float32")
                    sp_samples[code] = row["species"]
                site_labels.setdefault(code, {})[drug_name] = row["label"]
        codes = list(X_samples.keys())
        Xs.append(np.array([X_samples[c] for c in codes], dtype="float32"))
        sps.append(np.array([sp_samples[c] for c in codes]))
        Y = np.full((len(codes), len(DRUGS_10)), np.nan, dtype=float)
        for i, drug in enumerate(DRUGS_10):
            for j, code in enumerate(codes):
                if drug in site_labels.get(code, {}):
                    Y[j, i] = site_labels[code][drug]
        Ys.append(Y)
    X_all = np.concatenate(Xs); Y_all = np.concatenate(Ys); sp_all = np.concatenate(sps)

print(f"Aggregated: X={X_all.shape}  Y={Y_all.shape}  species={len(np.unique(sp_all))}")

In [ ]:
# ── Species-stratified 70/15/15 split + preprocessing ──
if SMOKE:
    rs = np.random.RandomState(SEED)
    idx = rs.choice(len(Y_all), size=min(1500, len(Y_all)), replace=False)
    X_all, Y_all, sp_all = X_all[idx], Y_all[idx], sp_all[idx]

idx = np.arange(len(Y_all)).reshape(-1, 1)
idx_trval, _, _, _ = stratified_species_drug_split(
    idx, np.zeros(len(Y_all)), species=sp_all, test_size=0.15, random_state=SEED)
idx_trval = idx_trval.flatten().astype(int)
idx_train, idx_val, _, _ = stratified_species_drug_split(
    idx_trval.reshape(-1, 1), np.zeros(len(idx_trval)), species=sp_all[idx_trval],
    test_size=0.15 / 0.85, random_state=SEED)
idx_train = idx_train.flatten().astype(int)
idx_val = idx_val.flatten().astype(int)

state_agg = fit_input_transform(X_all[idx_train], "log1p+standardize")
X_train_pp = apply_input_transform(X_all[idx_train], state_agg)
X_val_pp = apply_input_transform(X_all[idx_val], state_agg)
Y_train, Y_val = Y_all[idx_train], Y_all[idx_val]
print(f"train={X_train_pp.shape[0]}  val={X_val_pp.shape[0]}")

In [ ]:
# ── Train the aggregated shared MLP once, recording history ──
mlp_agg = MultiLabelMaldiMLP(
    hidden_dim=512, head_dims=(256, 128), use_attention=False,
    dropout_high=BEST["drop"], dropout_low=BEST["drop"] / 2.0,
    learning_rate=BEST["lr"], weight_decay=1e-4,
    batch_size=64,
    epochs=3 if SMOKE else 100,
    early_stopping_patience=2 if SMOKE else 15,
    warmup_epochs=0 if SMOKE else 10,
    val_fraction=0.1, random_state=SEED, verbose=True, device=DEVICE_NAME)
mlp_agg.fit(X_train_pp, Y_train)

hist = mlp_agg.history_

In [ ]:
# ── Plot: masked BCE loss + val macro Balanced Accuracy ──
hist = mlp_agg.history_
ep = np.arange(1, len(hist["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
ax.plot(ep, hist["train_loss"], label="train", lw=2)
ax.plot(ep, hist["val_loss"], label="val", lw=2)
best_ep = int(np.argmin(hist["val_loss"])) + 1
ax.axvline(best_ep, color="gray", ls=":", lw=1)
ax.annotate(f"best epoch {best_ep}", (best_ep, min(hist["val_loss"])),
            textcoords="offset points", xytext=(4, 6), fontsize=9, color="gray")
ax.set_title("Aggregated shared MLP — masked BCE loss")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.legend(); ax.grid(True, ls="--", alpha=0.4)

ax = axes[1]
ax.plot(ep, hist["val_macro_balacc"], color="#2ca02c", lw=2)
ax.axvline(best_ep, color="gray", ls=":", lw=1)
ax.set_title("Validation macro Balanced Accuracy @0.5")
ax.set_xlabel("Epoch"); ax.set_ylabel("macro BalAcc")
ax.grid(True, ls="--", alpha=0.4)

fig.suptitle("04 — aggregated multi-label training curves", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "04_loss_curves.pdf", bbox_inches="tight")
plt.show()

pd.DataFrame(hist).to_csv(OUT_DIR / "history_04.csv", index=False)
print("saved:", OUT_DIR / "04_loss_curves.pdf")

## What to look for

- **Loss vs. macro BalAcc**: validation CE can increase while macro BalAcc keeps
  rising (calibration drift, ranking preserved). The model actually used is the
  best-val-loss checkpoint, so look at both panels around the dotted epoch.
- Masked BCE averages over known labels only — samples tested for more drugs
  contribute more; compare the early-stop epoch with the per-drug models (02).